<link href="https://fonts.googleapis.com/css2?family=Oswald:wght@700&display=swap" rel="stylesheet">

<h1 style="
font-family: 'Oswald', sans-serif;
font-weight: 700;
font-style: italic;
font-size: 90px;
letter-spacing: 2px;
color: #E7C173;
text-shadow: 3px 3px 0 #333;
">
MACHINE LEARNING<br>IN INDUSTRY
</h1>

# DIY: Data Preprocessing and Feature Engineering

Practice encoding strategies, missing-value handling, scaling decisions, and feature engineering on **three OpenML datasets**, then add one **local retail-panel exercise** before finishing with a **capstone** where you turn one raw table into a usable feature matrix.

This notebook builds on what you learned in *Day 1 — Data Preprocessing and Feature Engineering* and ends with an end-to-end transfer exercise.

---

### Table of Contents

- [1. Load Datasets](#load)
- [2. Choosing Encodings for High-Cardinality Categories](#ohe-explosion)
- [3. Comparing Encodings on Train and Test](#ohe-overfitting)
- [4. How Should We Encode Zipcode?](#freq-collision)
- [5. Imputing MonthlyIncome](#mean-imputation)
- [6. Missingness and Prediction](#missingness-signal)
- [7. Handling Extreme Values](#clipping-pitfall)
- [8. Scaling Choices Under Outliers](#scaler-choice)
- [9. Dimensionality Reduction Before Prediction](#pca-pitfall)
- [10. Feature Engineering: Grouped Variables for Credit Scoring](#grouped-features)
- [11. Group and Date Features in a Retail Panel](#retail-context)
- [12. Capstone: From Raw Data to a Usable Feature Matrix](#capstone)


---

## Setup

In [ ]:
import os
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
print(f"Working directory: {Path.cwd()}")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ── helpers ──
def show(df, n=5):
    display(df.head(n))


def fetch_openml_compat(data_id: int):
    try:
        return fetch_openml(data_id=data_id, as_frame=True, parser="auto")
    except TypeError:
        return fetch_openml(data_id=data_id, as_frame=True)


SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

print("Environment ready.")

---

## <a id="load"></a> Section 1 — Load Datasets

We use three datasets from OpenML.

| Dataset | Rows | Features | Task | Description |
|---|---:|---:|---|---|
| **Amazon Employee Access** | ~32 K | 9 categorical | Binary classification (access granted/denied) | [OpenML 4135](https://www.openml.org/d/4135) |
| **King County House Sales** | ~21 K | 19 mixed | Regression (house price) | [OpenML 42092](https://www.openml.org/d/42092) |
| **Give Me Some Credit** | ~150 K | 10 numeric | Binary classification (serious financial distress) | [OpenML 45577](https://www.openml.org/d/45577) |
A later exercise also uses one **local retail panel** (`day1/generated/retail_panel_issues.csv`) because grouped and datetime features are more realistic there than in a purely cross-sectional table.


In [ ]:
# ── Amazon Employee Access (OpenML ID 4135) ──
amazon_raw = fetch_openml_compat(data_id=4135)
amazon = amazon_raw.frame

print("Amazon Employee Access")
print(f"  Shape: {amazon.shape}")
print(f"  Target: '{amazon_raw.target.name}' — positive rate: {amazon_raw.target.astype(int).mean():.4f}")
print()
show(amazon)

In [ ]:
# ── King County House Sales (OpenML ID 42092) ──
kc_raw = fetch_openml_compat(data_id=42092)
kc = kc_raw.frame

print("King County House Sales")
print(f"  Shape: {kc.shape}")
print(f"  Target: '{kc_raw.target.name}' — median: ${kc_raw.target.astype(float).median():,.0f}")
print()
show(kc)

In [ ]:
# ── Give Me Some Credit (OpenML ID 45577) ──
gmsc_raw = fetch_openml_compat(data_id=45577)
gmsc = gmsc_raw.frame

print("Give Me Some Credit")
print(f"  Shape: {gmsc.shape}")
print(f"  Target: '{gmsc_raw.target.name}' — positive rate: {gmsc_raw.target.astype(int).mean():.4f}")
print()
show(gmsc)

In [ ]:
# Your first inspection here


---

## <a id="ohe-explosion"></a> Exercise 2 — First Encoding Decisions (Amazon)

Treat Amazon as a new dataset you have just received.

Inspect the columns, decide what needs encoding, and use `RESOURCE` as one concrete test case.

> **Do It Yourself**
>
> 1. Inspect the Amazon columns and decide which ones need encoding.
> 2. Start with `RESOURCE` and decide which encoding to apply.
> 3. Then encode all the other non-numerical variables.
> 4. Write your own recommendation: when would you still use OHE here, and when would you switch to something else?

######

In [ ]:
# Your exploration here


---

### Quick Reference: Models and Metrics Used Below

The remaining exercises use simple models to **measure the impact of preprocessing choices**. You do not need to understand how these models work yet — that is the focus of Day 2. For now, just read the output numbers:

| Concept | What it is | How to read it |
|---|---|---|
| **AUC** (Area Under the ROC Curve) | Measures how well a classifier ranks positive vs negative examples | 0.5 = random guessing, 1.0 = perfect. Higher is better. |
| **R²** (R-squared) | Measures how much variance in the target a regression model explains | 0.0 = no better than predicting the mean, 1.0 = perfect. Can go negative if the model is worse than the mean. |
| **LogisticRegression** | A simple linear classifier | [sklearn docs](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) |
| **DecisionTreeClassifier** | A single decision tree classifier | [sklearn docs](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) |
| **KNeighborsClassifier** | A distance-based classifier | [sklearn docs](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) |
| **LinearRegression** | A simple linear regression model | [sklearn docs](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) |
| **GradientBoostingClassifier** | A tree-based ensemble that builds many small decision trees sequentially | [sklearn docs](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) |

The key takeaway in every exercise is **relative**: does the metric go up or down when we change the preprocessing? The absolute numbers matter less than the comparison.


---

## <a id="ohe-overfitting"></a> Exercise 3 — Comparing Encodings on Train and Test (Amazon)

Now compare **two models** while changing only how the high-cardinality `RESOURCE` feature is represented.

Brief definition: **overfitting** happens when a model learns patterns that help much more on the training set than on new data. In practice, this often appears as a large train/test performance gap (given the data distribution did not change between the two sets).

Run the cells below and compare what happens for both logistic regression and a decision tree.

> **Do It Yourself**
>
> Compare two models on the same train/test split:
> - `LogisticRegression`
> - `DecisionTreeClassifier`
>
> For each model, compare:
> - naive one-hot encoding (`OneHotEncoder(handle_unknown="ignore")`)
> - one-hot encoding that groups rare levels using `min_frequency` (see the lesson notebook for details on this parameter)
>
> Follow these steps:
>
> **Step 1 — Create two encoders**: one naive, one with `min_frequency` set (remember: when using `min_frequency`, set `handle_unknown="infrequent_if_exist"`)
>
> **Step 2 — Encode**: fit each encoder on `X_train`, transform both `X_train` and `X_test`
>
> **Step 3 — Fit each model and score with `roc_auc_score`** on train and test (use `predict_proba(...)[:, 1]`)
>
> Then answer:
> - How many columns does each encoding produce?
> - Does grouping rare levels reduce the train/test gap for both models?
> - Which model seems more sensitive to rare categories?

In [ ]:
feature_cols = [c for c in amazon.columns if c != amazon_raw.target.name]
y = amazon_raw.target.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    amazon[feature_cols], y, test_size=0.3, random_state=SEED, stratify=y
)

In [ ]:
# Step 1: Create two encoders
# ohe_naive = OneHotEncoder(...)
# ohe_grouped = OneHotEncoder(min_frequency=..., handle_unknown=...)

# Step 2: Fit on X_train, transform both X_train and X_test
# X_tr_naive = ohe_naive.fit_transform(X_train)
# X_te_naive = ohe_naive.transform(X_test)
# ... same for ohe_grouped ...

# Step 3: Fit models and compute roc_auc_score
# lr = LogisticRegression(C=100, max_iter=1000, solver="liblinear", random_state=SEED)
# lr.fit(X_tr_naive, y_train)
# train_auc = roc_auc_score(y_train, lr.predict_proba(X_tr_naive)[:, 1])
# test_auc = roc_auc_score(y_test, lr.predict_proba(X_te_naive)[:, 1])
# ... repeat for grouped, then for DecisionTreeClassifier ...

---

## <a id="freq-collision"></a> Exercise 4 — Encoding `zipcode` (King County)

Inspect the relationship between `zipcode`, how often each value appears, and how house prices vary across zipcodes.

Then decide whether frequency encoding looks acceptable.

> **Do It Yourself**
>
> 1. Inspect the relationship between zipcode frequency and median price.
> 2. Decide whether frequency encoding looks acceptable for `zipcode`.
> 3. Propose at least one alternative encoding that would preserve more information.
>
> You can try one or more of these:
> - **Target encoding**: replace each zipcode with the mean price in the training set.
> - **Ordinal encoding by target**: rank zipcodes by mean price and use the rank.
> - **Binning**: group zipcodes into price tiers and one-hot encode the tiers.
>
> *Hint: compute encodings on the training set only to avoid leakage.*

In [ ]:
show(kc, 2)

In [ ]:
# Your encoding experiment here


---

## <a id="mean-imputation"></a> Exercise 5 — Imputing `MonthlyIncome` (Give Me Some Credit)

Start by looking at the observed distribution before deciding on a fill rule.

The main question is whether one global number is a reasonable stand-in for every missing income value.

> **Do It Yourself**
>
> Try these alternatives to global mean imputation and compare the resulting distributions:
> - **Median imputation** — does it reduce the distortion compared to the mean?
> - **Group-conditional imputation** — impute with the median of the borrower's age group (e.g., 10-year bins). How does the filled distribution compare?
> - **Random-sample imputation** — for each missing value, sample randomly from observed values. Does this preserve the distribution shape better?
>
> *Think about: which method would you choose if your downstream model is a logistic regression? What if it is a gradient-boosted tree?*

In [ ]:
show(gmsc)

In [ ]:
# Your imputation experiment here


---

## <a id="missingness-signal"></a> Exercise 6 — Missingness and Prediction (Give Me Some Credit)

Before treating missing values as a nuisance, check whether the missingness pattern itself is informative.

The question here is: do rows with missing values behave differently from rows with observed values?

> **Do It Yourself**
>
> Explore further:
> - Add a **missing indicator** column for each feature with missing values. Train a `LogisticRegression` with and without the indicators — does test AUC improve?
> - Try `HistGradientBoostingClassifier`, which handles missing values natively (no imputation needed). Compare its AUC to the logistic regression with median imputation.
> - Does the missing indicator help more for `MonthlyIncome` or `NumberOfDependents`? Try adding them one at a time.
>
> *Think about: in a production pipeline, would you always add missing indicators, or only when you have evidence that missingness is informative?*

In [ ]:
show(gmsc, 2)

In [ ]:
# Your missingness experiment here


---

## <a id="clipping-pitfall"></a> Exercise 7 — Handling Extreme Values (Give Me Some Credit)

Extreme values often deserve attention, but not every extreme should be clipped away.

Start by inspecting how default risk changes across the range of `RevolvingUtilizationOfUnsecuredLines`. Then ask: if we cap everything above `1.0`, what information disappears?

> **Do It Yourself**
>
> 1. Summarize default rate for `RevolvingUtilizationOfUnsecuredLines` across a few utilization bands.
> 2. Compare the bands just below and above `1.0`.
> 3. Decide whether clipping at `1.0` would erase a meaningful risk regime.
> 4. If you want, suggest a safer alternative than blind clipping.
>
> *Hint: focus especially on the difference between `0.7–1.0` and `1.0–2.0`.*

In [ ]:
# Your clipping alternative experiment here


---

## <a id="scaler-choice"></a> Exercise 8 — Scaling Choices Under Outliers

Scaling is not just about putting variables on the same range. With distance-based models, the **choice of scaler** can change how much outliers distort the geometry of the problem.

Here you will compare four versions of the same `KNeighborsClassifier` on an outlier-heavy subset of **Give Me Some Credit**:
- no scaling
- `StandardScaler`
- `MinMaxScaler`
- `RobustScaler`

The goal is to see whether a scaler that is less sensitive to extreme values gives better generalization.


> **Do It Yourself**
>
> 1. Take a manageable sample of the data and keep a few numeric columns with heavy tails.
> 2. Train the same `KNeighborsClassifier` four times: no scaling, standard, min-max, robust.
> 3. Compare **train AUC**, **test AUC**, and the **train-test gap**.
> 4. Decide whether one scaler clearly handles the outliers better.
>
> Follow these steps:
>
> **Step 1 — Sample and split**: take ~8k rows, pick the outlier-heavy columns (`RevolvingUtilizationOfUnsecuredLines`, `DebtRatio`, `MonthlyIncome`, plus a couple of cleaner ones), and do a `train_test_split`.
>
> **Step 2 — Build a Pipeline for each scaler**: use the same pattern from the lesson notebook — `Pipeline([("imputer", SimpleImputer(...)), ("scaler", ...), ("model", KNeighborsClassifier(...))])`. Run it four times swapping only the scaler step (or skipping it for the "no scaling" case).
>
> **Step 3 — Score each**: `roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])` — same pattern as earlier exercises.
>
> *Hint: `RevolvingUtilizationOfUnsecuredLines`, `DebtRatio`, and `MonthlyIncome` are the columns to watch.*

In [ ]:
# Step 1: Sample and split
# scale_sample = gmsc.sample(n=8_000, random_state=SEED).copy()
# scale_features = ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio", "MonthlyIncome", "age", "NumberOfOpenCreditLinesAndLoans"]
# X_scale_train, X_scale_test, y_scale_train, y_scale_test = train_test_split(...)

# Step 2: Build a Pipeline for each scaler (same pattern as the lesson notebook)
# Example for one scaler:
# pipe = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("scaler", RobustScaler()),
#     ("model", KNeighborsClassifier(n_neighbors=25)),
# ])
# pipe.fit(X_scale_train, y_scale_train)

# Step 3: Score
# train_auc = roc_auc_score(y_scale_train, pipe.predict_proba(X_scale_train)[:, 1])
# test_auc = roc_auc_score(y_scale_test, pipe.predict_proba(X_scale_test)[:, 1])

# Repeat for: no scaler, StandardScaler, MinMaxScaler, RobustScaler
# Collect results into a DataFrame and display

---

## <a id="pca-pitfall"></a> Exercise 9 — Dimensionality Reduction Before Prediction

Before treating PCA as a harmless preprocessing step, check whether the retained high-variance directions are actually the predictive ones.

The key question is not just whether PCA compresses the data, but whether it preserves the information needed for prediction.

We demonstrate this with a small synthetic 2D dataset where the target lives mostly in a low-variance direction.

> **Do It Yourself**
>
> 1. Build a small comparison between PCA and a no-PCA baseline.
> 2. Inspect both the scores and the plot.
> 3. Write one short conclusion: when might unsupervised dimensionality reduction remove useful signal?

In [ ]:
# Your PCA experiment here


---

## <a id="grouped-features"></a> Exercise 10 — Feature Engineering: Grouped Variables for Credit Scoring

Raw features rarely tell the full story. In credit scoring, **ratios and interactions** between variables often carry more signal than individual columns.

For example, knowing someone's `DebtRatio` and `MonthlyIncome` separately is useful — but their product gives you `MonthlyDebt`, which directly measures the dollar burden on the borrower.

The Give Me Some Credit dataset has 10 features describing a borrower's financial profile. Your task is to **create new grouped/ratio features** and check whether they help a simple linear model.

Here are some ideas from top Kaggle solutions:

| New feature | Formula | Intuition |
|---|---|---|
| `MonthlyDebt` | `DebtRatio × MonthlyIncome` | Dollar amount of monthly debt obligations |
| `IncomePerDependent` | `MonthlyIncome / (NumberOfDependents + 1)` | How stretched is the household income? |
| `TotalLatePay` | `Num30-59 + Num60-89 + Num90+` | Aggregate delinquency count across all severity levels |
| `OverLimitFlag` | `RevolvingUtilization > 1.0` | Binary: is the borrower over their credit limit? |
| `HighDebtFlag` | `DebtRatio > 1.0` | Binary: do debts exceed income? |
| `LatePay_per_Line` | `TotalLatePay / (NumOpenLines + 1)` | Rate of delinquency relative to number of open accounts |

📚 [NYC Data Science Blog — Give Me Some Credit](https://nycdatascience.com/blog/student-works/kaggle-predict-consumer-credit-default/) · [Kaggle: Give Me Some Credit](https://www.kaggle.com/c/GiveMeSomeCredit)

> **Do It Yourself**
>
> Create at least **3 new grouped/ratio features** from the table above (or invent your own) and measure whether they improve a `LogisticRegression`.
>
> Follow these steps:
>
> **Step 1 — Baseline**: build a `Pipeline` with `SimpleImputer(strategy="median")` → `StandardScaler()` → `LogisticRegression()`. Fit on `X_tr`, score on `X_te` with `roc_auc_score`. This is your baseline AUC.
>
> **Step 2 — Engineer features**: add new columns to copies of `X_tr` and `X_te`. Some ideas from the table:
> - `TotalLatePay = NumberOfTime30-59DaysPastDueNotWorse + NumberOfTime60-89DaysPastDueNotWorse + NumberOfTimes90DaysLate`
> - `MonthlyDebt = DebtRatio * MonthlyIncome`
> - `IncomePerDependent = MonthlyIncome / (NumberOfDependents + 1)`
>
> **Step 3 — Fit the same pipeline** on the enriched data and compare test AUC to baseline.
>
> **Step 4 — Inspect coefficients**: look at `pipe.named_steps["logisticregression"].coef_` to see if engineered features rank in the top 10 by absolute magnitude.
>
> *Think about: linear models cannot invent arbitrary ratios or thresholds by themselves. Does feature engineering help more here than it would for a tree model?*

In [ ]:
# Step 1: Baseline pipeline (same pattern as the lesson notebook)
# baseline_pipe = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("scaler", StandardScaler()),
#     ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
# ])
# baseline_pipe.fit(X_tr, y_tr)
# auc_baseline = roc_auc_score(y_te, baseline_pipe.predict_proba(X_te)[:, 1])

# Step 2: Add engineered features
# X_tr_eng = X_tr.copy()
# X_te_eng = X_te.copy()
# for df in [X_tr_eng, X_te_eng]:
#     df["TotalLatePay"] = df["NumberOfTime30-59DaysPastDueNotWorse"] + ...
#     df["MonthlyDebt"] = df["DebtRatio"] * df["MonthlyIncome"]
#     df["IncomePerDependent"] = df["MonthlyIncome"] / (df["NumberOfDependents"] + 1)

# Step 3: Fit same pipeline on enriched data
# eng_pipe = Pipeline([...])  # same as baseline
# eng_pipe.fit(X_tr_eng, y_tr)
# auc_eng = roc_auc_score(y_te, eng_pipe.predict_proba(X_te_eng)[:, 1])

# Step 4: Inspect coefficients
# coefs = pd.Series(
#     np.abs(eng_pipe.named_steps["model"].coef_[0]),
#     index=X_tr_eng.columns,
# ).sort_values(ascending=False)
# display(coefs.head(10))

---

## <a id="retail-context"></a> Exercise 11 — Group and Date Features in a Retail Panel

A raw row often misses context. In panel data, the same store can behave differently across time, and a raw numeric identifier such as `store_id` is usually **not** the feature you really want.

In this exercise, compare four versions of the same `LinearRegression` on a local retail panel:
- base operational features only
- base features + raw `store_id`
- base features + a **train-only** store-level aggregate
- base features + the store aggregate + simple date-derived features

The goal is to see whether contextual features help more than the raw identifier itself.


> **Do It Yourself**
>
> 1. Load `day1/generated/retail_panel_issues.csv`, drop rows where `sales` is missing, sort by date.
> 2. Use a **chronological split** (first 70% by date = train, rest = test).
> 3. Compare four `LinearRegression` models with different feature sets, reporting **train R²**, **test R²**, and **gap**.
> 4. Decide what this tells you about raw IDs, grouped features, and date features.
>
> Follow these steps:
>
> **Step 1 — Load and split chronologically**: sort by date, take the first 70% as train. This is different from `train_test_split` — here order matters.
>
> **Step 2 — Define base features**: start with `["inventory_units", "promotion_discount_pct", "store_traffic_index"]`.
>
> **Step 3 — Add features incrementally** and fit a `Pipeline([("imputer", SimpleImputer(...)), ("model", LinearRegression())])` for each variant:
> - **(a)** base features only
> - **(b)** base + raw `store_id`
> - **(c)** base + `store_mean_sales_train` (compute `groupby("store_id")["sales"].mean()` on **train only**, then `.map()` onto both splits — same pattern as the group aggregates in the lesson notebook)
> - **(d)** base + store mean + date features (`dt.month`, `dt.dayofweek` — same pattern as the datetime cell in the lesson notebook)
>
> **Step 4 — Score with `r2_score`** on both train and test for each variant.
>
> *Hint: if you compute a store-level sales aggregate, learn it on the training period only.*

In [ ]:
# Step 1: Load and chronological split
# retail = pd.read_csv("day1/generated/retail_panel_issues.csv")
# retail = retail[retail["sales"].notna()].copy()
# retail["date"] = pd.to_datetime(retail["date"])
# retail = retail.sort_values("date").reset_index(drop=True)
# split_idx = int(len(retail) * 0.7)
# retail_train = retail.iloc[:split_idx].copy()
# retail_test = retail.iloc[split_idx:].copy()

# Step 2: Base features
# base_cols = ["inventory_units", "promotion_discount_pct", "store_traffic_index"]

# Step 3a: Group aggregate (train only, then map — same pattern as lesson notebook)
# store_mean_sales = retail_train.groupby("store_id")["sales"].mean()
# global_mean = retail_train["sales"].mean()
# for df in [retail_train, retail_test]:
#     df["store_mean_sales_train"] = df["store_id"].map(store_mean_sales).fillna(global_mean)

# Step 3b: Date features (same pattern as lesson notebook)
# for df in [retail_train, retail_test]:
#     df["sale_month"] = df["date"].dt.month
#     df["sale_dayofweek"] = df["date"].dt.dayofweek

# Step 4: For each feature set, fit Pipeline and score with r2_score
# pipe = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("model", LinearRegression()),
# ])
# pipe.fit(retail_train[feature_cols], retail_train["sales"])
# train_r2 = r2_score(retail_train["sales"], pipe.predict(retail_train[feature_cols]))
# test_r2 = r2_score(retail_test["sales"], pipe.predict(retail_test[feature_cols]))

---

## <a id="capstone"></a> Exercise 12 — Capstone: From Raw Data to a Usable Feature Matrix

This final exercise shifts from isolated pitfalls to an end-to-end preprocessing workflow.

You will work on one **local dataset**: `day1/generated/king_county_capstone.csv`.

Your goal is to turn the raw table into a **numeric, leakage-safe feature matrix suitable for a scaling-sensitive model** such as a regularized linear model.

That means:
- no target or non-feature columns inside the final matrix
- no raw string/object columns left
- no accidental use of `val` or `test` to fit imputers, encoders, or scalers
- a documented policy for missing values, categoricals, invalid values, date features, grouped features, and scaling

Date-derived and grouped features are encouraged here.

In [ ]:
# ── Load capstone dataset ──
capstone = pd.read_csv("day1/generated/king_county_capstone.csv", parse_dates=["date"])

print("King County capstone")
print(f"  Shape: {capstone.shape}")
print(f"  Splits: {capstone['split'].value_counts().sort_index().to_dict()}")
print(f"  Target: 'price' — median: ${capstone['price'].median():,.0f}")
print()

missing_summary = (
    capstone.isna().mean().mul(100).rename("missing_pct").sort_values(ascending=False).to_frame()
)
print("Columns with missing values:")
display(missing_summary.query("missing_pct > 0"))
print()
print("Dtypes:")
display(capstone.dtypes.rename("dtype").to_frame())
print()
show(capstone)

> **Capstone Task**
>
> Starting from the raw dataframe above, produce:
>
> 1. A preprocessing policy table with columns such as `column_or_group`, `issue`, `choice`, `why`.
> 2. A clean train / val / test split that uses the provided `split` column.
> 3. Final feature matrices named `X_train_final`, `X_val_final`, `X_test_final`.
> 4. Matching targets named `y_train`, `y_val`, `y_test`.
> 5. A short log of engineered features you created from dates or grouped relationships.
>
> Constraints:
> - Assume the downstream model is **scale-sensitive**, so your final matrix should be appropriate for a linear model.
> - Fit every learned step on `train` only.
> - Do not use `price`, `split`, IDs, or process metadata as features.
> - Handle unseen categories safely.
> - End with a matrix that is numeric and has no unintended missing values.

In [ ]:
# Your capstone work starts here

# Suggested variable names:
# train_cap = capstone.loc[capstone["split"] == "train"].copy()
# val_cap = capstone.loc[capstone["split"] == "val"].copy()
# test_cap = capstone.loc[capstone["split"] == "test"].copy()
# y_train = train_cap["price"].copy()
# y_val = val_cap["price"].copy()
# y_test = test_cap["price"].copy()
# X_train_final = ...
# X_val_final = ...
# X_test_final = ...

In [ ]:
# Optional self-checks after you build the final matrices

# assert list(X_train_final.columns) == list(X_val_final.columns) == list(X_test_final.columns)
# assert X_train_final.select_dtypes(exclude=[np.number]).shape[1] == 0
# assert X_val_final.select_dtypes(exclude=[np.number]).shape[1] == 0
# assert X_test_final.select_dtypes(exclude=[np.number]).shape[1] == 0
# assert not X_train_final.isna().any().any()
# assert not X_val_final.isna().any().any()
# assert not X_test_final.isna().any().any()
# print("Capstone matrices look model-ready.")

---

## Scratch Space

In [ ]:
# Free exploration
